In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ROCKET-CRANE (paper-style) with YOUR SAME I/O FORMAT

✅ Input:  DIMACS-like .d file with lines:
      a <u> <v> <w> <...ignored...>
   - Aggregates parallel arcs (u,v) by summing weights
   - Deterministic node mapping (sorted node ids)

✅ Output: ranking CSV with columns EXACTLY:
      ["Node ID", "Order"]
   (sorted by Order ascending)

✅ NO log file written anymore.
✅ Prints important info to screen:
     - n,m
     - after Rocket ratio + time
     - after Crane ratio + time
     - final ratio + total time
✅ Filenames include method name: "rocketcrane"

Dependencies: numpy, pandas
"""

import os
import time
import math
import random
from collections import defaultdict
import numpy as np
import pandas as pd


# ============================================================
# 1) DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Scoring: forward/backward weight for a ranking
# ============================================================

def compute_forward_backward(edges_indexed, rank):
    # rank: list/array length n, rank[i] is position (0..n-1)
    total_w = 0.0
    fw = 0.0
    for u, v, w in edges_indexed:
        total_w += w
        if rank[u] < rank[v]:
            fw += w
    bw = total_w - fw
    return total_w, fw, bw


# ============================================================
# 3) Output: EXACT same CSV schema as your other code
# ============================================================

def write_ranking_csv_nodeid_order(path, index_to_node, rank):
    n = len(rank)
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(path, index=False)


# ============================================================
# 4) Rocket: continuous surrogate + Adam
# ============================================================

def rocket_optimize(
    n,
    eu, ev, ew,
    beta=1.0,
    iters=200,
    lr=0.05,
    seed=1,
    deadline=None,
):
    """
    Optimize positions P ∈ R^n to maximize smooth surrogate:
        sum_e w_e * sigmoid(beta*(P[v]-P[u]))
    using Adam.

    Returns: best_perm (order of vertices), best_rank (position array), best_fw
    """
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    # normalize weights for stability
    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    # Adam state
    m = np.zeros(n, dtype=np.float32)
    v = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def perm_from_positions(Pvec):
        return np.argsort(Pvec, kind="mergesort").astype(np.int32)

    def rank_from_perm(perm):
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    # best snapshot by TRUE discrete objective (forward weight)
    perm0 = perm_from_positions(P)
    rank0 = rank_from_perm(perm0)
    fw_best = float(np.sum(ew[rank0[eu] < rank0[ev]]))
    perm_best = perm0.copy()
    rank_best = rank0.copy()

    for it in range(1, iters + 1):
        if deadline is not None and time.time() >= deadline:
            break

        d = (P[ev] - P[eu]).astype(np.float32)
        x = beta * d
        sig = 1.0 / (1.0 + np.exp(-x, dtype=np.float32))

        # maximize sum(w_hat*sig) -> minimize -sum(w_hat*sig)
        gfac = (beta * w_hat * sig * (1.0 - sig)).astype(np.float32)

        grad = np.zeros(n, dtype=np.float32)
        np.add.at(grad, eu, +gfac)
        np.add.at(grad, ev, -gfac)

        # Adam step
        t += 1
        m = (b1 * m + (1 - b1) * grad).astype(np.float32)
        v = (b2 * v + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m / (1 - (b1 ** t))
        vhat = v / (1 - (b2 ** t))
        P = (P - lr * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # evaluate discrete objective
        perm = perm_from_positions(P)
        rank = rank_from_perm(perm)
        fw = float(np.sum(ew[rank[eu] < rank[ev]]))
        if fw > fw_best + 1e-9:
            fw_best = fw
            perm_best = perm.copy()
            rank_best = rank.copy()

    return perm_best, rank_best, fw_best


# ============================================================
# 5) Crane: TopoShuffle init + SA + greedy swaps
# ============================================================

def build_inc_out_lists(n, eu, ev, ew):
    out_nbrs = [[] for _ in range(n)]
    out_w = [[] for _ in range(n)]
    in_nbrs = [[] for _ in range(n)]
    in_w = [[] for _ in range(n)]
    for u, v, w in zip(eu.tolist(), ev.tolist(), ew.tolist()):
        out_nbrs[u].append(v); out_w[u].append(w)
        in_nbrs[v].append(u);  in_w[v].append(w)
    out_nbrs = [np.array(x, dtype=np.int32) for x in out_nbrs]
    out_w = [np.array(x, dtype=np.float32) for x in out_w]
    in_nbrs = [np.array(x, dtype=np.int32) for x in in_nbrs]
    in_w = [np.array(x, dtype=np.float32) for x in in_w]
    return out_nbrs, out_w, in_nbrs, in_w


def toposhuffle_init(n, perm, eu, ev, seed=1):
    rng = random.Random(seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    out = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in zip(eu.tolist(), ev.tolist()):
        if rank[u] < rank[v]:
            out[u].append(v)
            indeg[v] += 1

    zeros = [i for i in range(n) if indeg[i] == 0]
    rng.shuffle(zeros)

    order = []
    while zeros:
        u = zeros.pop()
        order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                zeros.append(v)
                j = rng.randrange(len(zeros))
                zeros[-1], zeros[j] = zeros[j], zeros[-1]

    if len(order) != n:
        seen = set(order)
        for i in range(n):
            if i not in seen:
                order.append(i)

    return np.array(order, dtype=np.int32)


def delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w):
    ra = int(rank[a])
    rb = int(rank[b])
    if ra == rb:
        return 0.0

    def r_after(x):
        if x == a:
            return rb
        if x == b:
            return ra
        return int(rank[x])

    delta = 0.0

    for u in (a, b):
        ru0 = int(rank[u])
        ru1 = r_after(u)
        nbrs = out_nbrs[u]
        ws = out_w[u]
        if nbrs.size:
            rv0 = rank[nbrs]
            rv1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[ru0 < rv0]))
            after = float(np.sum(ws[ru1 < rv1]))
            delta += (after - before)

    for u in (a, b):
        ru0 = int(rank[u])
        ru1 = r_after(u)
        nbrs = in_nbrs[u]
        ws = in_w[u]
        if nbrs.size:
            rx0 = rank[nbrs]
            rx1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[rx0 < ru0]))
            after = float(np.sum(ws[rx1 < ru1]))
            delta += (after - before)

    def direct_w(u, v):
        nbrs = out_nbrs[u]
        ws = out_w[u]
        if nbrs.size == 0:
            return 0.0
        idx = np.where(nbrs == v)[0]
        if idx.size == 0:
            return 0.0
        return float(np.sum(ws[idx]))

    wab = direct_w(a, b)
    wba = direct_w(b, a)

    if wab != 0.0:
        before = wab if ra < rb else 0.0
        after = wab if rb < ra else 0.0
        delta -= (after - before)

    if wba != 0.0:
        before = wba if rb < ra else 0.0
        after = wba if ra < rb else 0.0
        delta -= (after - before)

    return delta


def crane_refine(
    perm_init,
    eu, ev, ew,
    out_nbrs, out_w, in_nbrs, in_w,
    seed=2,
    sa_steps=20000,
    T0=1.0,
    alpha=0.95,
    mc_steps=20000,
    deadline=None,
):
    rng = random.Random(seed)
    n = len(perm_init)

    perm = toposhuffle_init(n, perm_init, eu, ev, seed=seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    best_perm = perm.copy()
    best_rank = rank.copy()
    best_fw = fw

    T = float(T0)
    cur_fw = fw

    for _ in range(1, sa_steps + 1):
        if deadline is not None and time.time() >= deadline:
            break

        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue

        a = int(perm[a_pos])
        b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)

        accept = (d >= 0) or (T > 1e-12 and rng.random() < math.exp(d / T))
        if accept:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)

            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

        T *= alpha

    perm = best_perm.copy()
    rank = best_rank.copy()
    cur_fw = best_fw

    for _ in range(1, mc_steps + 1):
        if deadline is not None and time.time() >= deadline:
            break

        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue

        a = int(perm[a_pos])
        b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)

        if d > 0:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)

            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

    return best_perm, best_rank, best_fw


# ============================================================
# 6) Driver (prints to screen; no log file)
# ============================================================

def run_rocketcrane(
    dimacs_path,
    out_csv_path,
    rng_seed=1,
    time_limit_s=None,
    rocket_iters=250,
    rocket_beta=1.0,
    rocket_lr=0.05,
    crane_sa_steps=25000,
    crane_T0=1.0,
    crane_alpha=0.95,
    crane_mc_steps=25000,
):
    t0 = time.time()
    deadline = (t0 + float(time_limit_s)) if (time_limit_s is not None and time_limit_s > 0) else None

    # Load graph
    t_read0 = time.time()
    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(dimacs_path)
    n = len(node_to_index)
    m = len(edges_indexed)
    t_read = time.time() - t_read0

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges_indexed):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    tot_w = float(np.sum(ew))

    # Build adjacency for Crane
    t_adj0 = time.time()
    out_nbrs, out_w, in_nbrs, in_w = build_inc_out_lists(n, eu, ev, ew)
    t_adj = time.time() - t_adj0

    print("\n================= ROCKET-CRANE RUN =================")
    print(f"Input:        {dimacs_path}")
    print(f"Seed:         {rng_seed}")
    print(f"n={n}  m={m}  total_w={tot_w:.6f}")
    print(f"read_s={t_read:.3f}  adj_build_s={t_adj:.3f}")
    if time_limit_s is not None:
        print(f"time_limit_s={time_limit_s}")
    print("----------------------------------------------------")

    # Rocket
    tR0 = time.time()
    perm_r, rank_r, fw_r = rocket_optimize(
        n=n, eu=eu, ev=ev, ew=ew,
        beta=rocket_beta,
        iters=rocket_iters,
        lr=rocket_lr,
        seed=rng_seed,
        deadline=deadline,
    )
    tR = time.time() - tR0
    tot1, fw1, bw1 = compute_forward_backward(edges_indexed, rank_r.tolist())
    ratio1 = 100.0 * fw1 / tot1 if tot1 > 0 else 0.0
    print(f"[rocket] fw={fw1:.6f}  ratio={ratio1:.6f}%  time_s={tR:.3f}")

    # Crane
    tC0 = time.time()
    perm_c, rank_c, fw_c = crane_refine(
        perm_init=perm_r,
        eu=eu, ev=ev, ew=ew,
        out_nbrs=out_nbrs, out_w=out_w,
        in_nbrs=in_nbrs, in_w=in_w,
        seed=rng_seed + 1,
        sa_steps=crane_sa_steps,
        T0=crane_T0,
        alpha=crane_alpha,
        mc_steps=crane_mc_steps,
        deadline=deadline,
    )
    tC = time.time() - tC0
    tot2, fw2, bw2 = compute_forward_backward(edges_indexed, rank_c.tolist())
    ratio2 = 100.0 * fw2 / tot2 if tot2 > 0 else 0.0
    print(f"[crane]  fw={fw2:.6f}  ratio={ratio2:.6f}%  time_s={tC:.3f}")

    # choose best of rocket vs rocket+crane (never output worse than rocket)
    if fw2 >= fw1 - 1e-9:
        final_rank = rank_c
        final_fw = fw2
        final_ratio = ratio2
        chosen = "rocket+crane"
    else:
        final_rank = rank_r
        final_fw = fw1
        final_ratio = ratio1
        chosen = "rocket_only"

    # write CSV
    write_ranking_csv_nodeid_order(out_csv_path, index_to_node, final_rank.tolist())

    elapsed = time.time() - t0
    print("----------------------------------------------------")
    print(f"[final] chosen={chosen}")
    print(f"[final] fw={final_fw:.6f}  ratio={final_ratio:.6f}%")
    print(f"[final] out_csv={out_csv_path}")
    print(f"[final] total_time_s={elapsed:.3f}")
    print("====================================================\n")


# ============================================================
# 7) Main (YOUR STYLE: input set inside __main__)
# ============================================================

if __name__ == "__main__":
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    method = "rocketcrane"
    base = edge_file[:-2] if edge_file.endswith(".d") else edge_file
    stamp = time.strftime("%Y%m%d-%H%M%S")
    out_csv = f"{base}_{method}_ranking_{stamp}.csv"

    run_rocketcrane(
        dimacs_path=edge_file,
        out_csv_path=out_csv,
        rng_seed=1,
        time_limit_s=None,        # set e.g. 120.0 for 2 minutes cap
        rocket_iters=250,
        rocket_beta=1.0,
        rocket_lr=0.05,
        crane_sa_steps=25000,
        crane_T0=1.0,
        crane_alpha=0.95,
        crane_mc_steps=25000,
    )



================= ROCKET-CRANE RUN =================
Input:        /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
Seed:         1
n=136648  m=5657719  total_w=41912140.000000
read_s=11.287  adj_build_s=5.283
----------------------------------------------------
[rocket] fw=33141851.000000  ratio=79.074584%  time_s=39.943
[crane]  fw=33428713.000000  ratio=79.759020%  time_s=7.472
----------------------------------------------------
[final] chosen=rocket+crane
[final] fw=33428713.000000  ratio=79.759020%
[final] out_csv=/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome_rocketcrane_ranking_20260208-120536.csv
[final] total_time_s=67.523



In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SEKE-ablated Rocket–Crane (cycle-aware) with SAME input/output format (.d -> ranking CSV)

Updated goals (per your request):
  ✅ Much less console spam:
        - No per-iteration Rocket/Crane prints
        - Only compact per-run summary (one block) + final aggregate ablation summary
        - Detailed info still goes to log files
  ✅ Stronger impact assessment of adding SEKE:
        - Run paired ablation across MULTIPLE seeds (same budgets, same file)
        - Produce a per-seed delta table + aggregate mean/std and win-rate
        - Write a single CSV summary of all runs + a compare log

Still keeps:
  - Deterministic DIMACS aggregation
  - SAME output CSV schema: columns ["Node ID", "Order"]
  - Per-lambda ranking CSV per seed (no timestamps)
  - Baseline run is lambda=0, SEKE run is lambda=LAM_SEKE
"""

import os
import time
import math
import random
from collections import defaultdict
import numpy as np
import pandas as pd


# ============================================================
# 0) Small helpers
# ============================================================

def ratio_percent(fw, total_w):
    return (100.0 * fw / total_w) if total_w > 0 else 0.0


def summarize_line(label, tot, fw, bw, fw_prev=None):
    if fw_prev is None:
        return f"[{label}] FW={fw:.6f}  BW={bw:.6f}  ratio={ratio_percent(fw, tot):.6f}%"
    else:
        return f"[{label}] FW={fw:.6f}  BW={bw:.6f}  ratio={ratio_percent(fw, tot):.6f}%  (ΔFW={fw-fw_prev:+.6f})"


def fmt_lam(lam: float):
    s = f"{lam:.6g}"
    return s.replace(".", "p").replace("-", "m")


def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)
    return path


# ============================================================
# 1) DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Scoring: forward/backward weight for a ranking
# ============================================================

def compute_forward_backward(edges_indexed, rank_arr):
    total_w = 0.0
    fw = 0.0
    for u, v, w in edges_indexed:
        total_w += w
        if rank_arr[u] < rank_arr[v]:
            fw += w
    bw = total_w - fw
    return total_w, fw, bw


# ============================================================
# 3) Output: EXACT same CSV schema as your other code
# ============================================================

def write_ranking_csv_nodeid_order(path, index_to_node, rank_arr):
    n = len(rank_arr)
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank_arr[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(path, index=False)


# ============================================================
# 4) SEKE suspiciousness q_e  (static)
# ============================================================

def build_seke_edge_suspiciousness(n: int, eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
                                  k: float = 20.0, tau=None, eps: float = 1e-12):
    Win = np.zeros(n, dtype=np.float64)
    Wout = np.zeros(n, dtype=np.float64)
    np.add.at(Wout, eu, ew.astype(np.float64))
    np.add.at(Win,  ev, ew.astype(np.float64))

    denom = Win[eu].astype(np.float64) + Wout[ev].astype(np.float64) + eps
    s = (ew.astype(np.float64) / denom)

    if tau is None:
        tau = float(np.median(s))

    x = float(k) * (s - float(tau))
    q = 1.0 / (1.0 + np.exp(-x))
    return q.astype(np.float32), float(tau)


# ============================================================
# 5) Rocket (cycle-aware SEKE term): Adam on combined objective
#     Obj = Σ ŵ σ(βΔ) + λ Σ ŵ q σ(-βΔ)
# ============================================================

def rocket_optimize_seke_cycleaware(
    n,
    eu, ev, ew,
    q_seke,                  # in (0,1)
    lambda_seke=0.0,
    beta=1.0,
    iters=300,
    lr=0.05,
    seed=1,
    deadline=None,
    logf=None,
    log_every=50,           # log file frequency (not console)
):
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    lam = float(lambda_seke)
    bet = float(beta)

    # Adam
    m = np.zeros(n, dtype=np.float32)
    v = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def perm_from_positions(Pvec):
        return np.argsort(Pvec, kind="mergesort").astype(np.int32)

    def rank_from_perm(perm):
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    perm0 = perm_from_positions(P)
    rank0 = rank_from_perm(perm0)
    fw_best = float(np.sum(ew[rank0[eu] < rank0[ev]]))
    perm_best = perm0.copy()
    rank_best = rank0.copy()

    if logf:
        logf.write(f"[rocket] lambda={lam:.6g} init_best_fw={fw_best:.6f}\n")
        logf.flush()

    for it in range(1, iters + 1):
        if deadline is not None and time.time() >= deadline:
            if logf:
                logf.write(f"[rocket] stop(deadline) it={it}\n")
                logf.flush()
            break

        d = (P[ev] - P[eu]).astype(np.float32)
        x = bet * d
        sig_f = 1.0 / (1.0 + np.exp(-x, dtype=np.float32))
        sigp = (sig_f * (1.0 - sig_f)).astype(np.float32)

        # d/dx [ ŵ*σ(x) + λ*ŵ*q*σ(-x) ] = ŵ*(1 - λ*q)*σ'(x)
        edge_grad_x = (w_hat * (1.0 - lam * q_seke) * sigp * bet).astype(np.float32)

        grad = np.zeros(n, dtype=np.float32)
        np.add.at(grad, eu, +edge_grad_x)
        np.add.at(grad, ev, -edge_grad_x)

        t += 1
        m = (b1 * m + (1 - b1) * grad).astype(np.float32)
        v = (b2 * v + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m / (1 - (b1 ** t))
        vhat = v / (1 - (b2 ** t))
        P = (P - float(lr) * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # discrete FW checkpoint
        perm = perm_from_positions(P)
        rank = rank_from_perm(perm)
        fw = float(np.sum(ew[rank[eu] < rank[ev]]))

        if fw > fw_best + 1e-9:
            fw_best = fw
            perm_best = perm.copy()
            rank_best = rank.copy()

        if logf and log_every and (it % log_every == 0 or it == 1 or it == iters):
            logf.write(f"[rocket] it={it} cur_fw={fw:.6f} best_fw={fw_best:.6f}\n")
            logf.flush()

    return perm_best, rank_best, fw_best


# ============================================================
# 6) Crane: TopoShuffle init + SA + greedy swaps
# ============================================================

def build_inc_out_lists(n, eu, ev, ew):
    out_nbrs = [[] for _ in range(n)]
    out_w = [[] for _ in range(n)]
    in_nbrs = [[] for _ in range(n)]
    in_w = [[] for _ in range(n)]
    for u, v, w in zip(eu.tolist(), ev.tolist(), ew.tolist()):
        out_nbrs[u].append(v); out_w[u].append(w)
        in_nbrs[v].append(u);  in_w[v].append(w)
    out_nbrs = [np.array(x, dtype=np.int32) for x in out_nbrs]
    out_w = [np.array(x, dtype=np.float32) for x in out_w]
    in_nbrs = [np.array(x, dtype=np.int32) for x in in_nbrs]
    in_w = [np.array(x, dtype=np.float32) for x in in_w]
    return out_nbrs, out_w, in_nbrs, in_w


def toposhuffle_init(n, perm, eu, ev, seed=1):
    rng = random.Random(seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    out = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in zip(eu.tolist(), ev.tolist()):
        if rank[u] < rank[v]:
            out[u].append(v)
            indeg[v] += 1

    zeros = [i for i in range(n) if indeg[i] == 0]
    rng.shuffle(zeros)

    order = []
    while zeros:
        u = zeros.pop()
        order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                zeros.append(v)
                j = rng.randrange(len(zeros))
                zeros[-1], zeros[j] = zeros[j], zeros[-1]

    if len(order) != n:
        seen = set(order)
        for i in range(n):
            if i not in seen:
                order.append(i)

    return np.array(order, dtype=np.int32)


def delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w):
    ra = int(rank[a])
    rb = int(rank[b])
    if ra == rb:
        return 0.0

    def r_after(x):
        if x == a:
            return rb
        if x == b:
            return ra
        return int(rank[x])

    delta = 0.0

    for u in (a, b):
        ru0 = int(rank[u])
        ru1 = r_after(u)
        nbrs = out_nbrs[u]
        ws = out_w[u]
        if nbrs.size:
            rv0 = rank[nbrs]
            rv1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[ru0 < rv0]))
            after = float(np.sum(ws[ru1 < rv1]))
            delta += (after - before)

    for u in (a, b):
        ru0 = int(rank[u])
        ru1 = r_after(u)
        nbrs = in_nbrs[u]
        ws = in_w[u]
        if nbrs.size:
            rx0 = rank[nbrs]
            rx1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[rx0 < ru0]))
            after = float(np.sum(ws[rx1 < ru1]))
            delta += (after - before)

    def direct_w(u, v):
        nbrs = out_nbrs[u]
        ws = out_w[u]
        if nbrs.size == 0:
            return 0.0
        idx = np.where(nbrs == v)[0]
        if idx.size == 0:
            return 0.0
        return float(np.sum(ws[idx]))

    wab = direct_w(a, b)
    wba = direct_w(b, a)

    if wab != 0.0:
        before = wab if ra < rb else 0.0
        after = wab if rb < ra else 0.0
        delta -= (after - before)

    if wba != 0.0:
        before = wba if rb < ra else 0.0
        after = wba if ra < rb else 0.0
        delta -= (after - before)

    return delta


def crane_refine(
    perm_init,
    eu, ev, ew,
    out_nbrs, out_w, in_nbrs, in_w,
    seed=2,
    sa_steps=25000,
    T0=1.0,
    alpha=0.95,
    mc_steps=25000,
    deadline=None,
    logf=None,
    sa_log_every=5000,     # log file frequency
    mc_log_every=5000,     # log file frequency
):
    rng = random.Random(seed)
    n = len(perm_init)

    perm = toposhuffle_init(n, perm_init, eu, ev, seed=seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    best_perm = perm.copy()
    best_rank = rank.copy()
    best_fw = fw

    if logf:
        logf.write(f"[crane] after_toposhuffle_fw={best_fw:.6f}\n")
        logf.flush()

    # SA
    T = float(T0)
    cur_fw = fw

    for it in range(1, sa_steps + 1):
        if deadline is not None and time.time() >= deadline:
            if logf:
                logf.write(f"[crane][sa] stop(deadline) it={it}\n")
                logf.flush()
            break

        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue

        a = int(perm[a_pos])
        b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)

        accept = (d >= 0) or (T > 1e-12 and rng.random() < math.exp(d / T))
        if accept:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)

            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

        T *= alpha

        if logf and sa_log_every and (it % sa_log_every == 0 or it == 1 or it == sa_steps):
            logf.write(f"[crane][sa] it={it} cur_fw={cur_fw:.6f} best_fw={best_fw:.6f} T={T:.4g}\n")
            logf.flush()

    # MC greedy from best
    perm = best_perm.copy()
    rank = best_rank.copy()
    cur_fw = best_fw

    for it in range(1, mc_steps + 1):
        if deadline is not None and time.time() >= deadline:
            if logf:
                logf.write(f"[crane][mc] stop(deadline) it={it}\n")
                logf.flush()
            break

        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue

        a = int(perm[a_pos])
        b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)

        if d > 0:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)

            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

        if logf and mc_log_every and (it % mc_log_every == 0 or it == 1 or it == mc_steps):
            logf.write(f"[crane][mc] it={it} cur_fw={cur_fw:.6f} best_fw={best_fw:.6f}\n")
            logf.flush()

    return best_perm, best_rank, best_fw


# ============================================================
# 7) One run (given seed+lambda): returns metrics for comparison
# ============================================================

def run_one_lambda(
    dimacs_path: str,
    out_csv_path: str,
    out_log_path: str,
    rng_seed=1,
    time_limit_s=None,

    # SEKE params
    seke_k=20.0,
    seke_tau=None,
    lambda_seke=0.0,

    # Rocket params
    rocket_iters=300,
    rocket_beta=1.0,
    rocket_lr=0.05,
    rocket_log_every=50,

    # Crane params
    crane_sa_steps=25000,
    crane_T0=1.0,
    crane_alpha=0.95,
    crane_mc_steps=25000,
    crane_sa_log_every=5000,
    crane_mc_log_every=5000,

    # Console control
    print_compact=True,
):
    t0 = time.time()
    deadline = (t0 + float(time_limit_s)) if (time_limit_s is not None and time_limit_s > 0) else None

    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(dimacs_path)
    n = len(node_to_index)
    m = len(edges_indexed)

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges_indexed):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    out_nbrs, out_w, in_nbrs, in_w = build_inc_out_lists(n, eu, ev, ew)

    q_seke, tau_used = build_seke_edge_suspiciousness(
        n=n, eu=eu, ev=ev, ew=ew,
        k=float(seke_k),
        tau=seke_tau
    )

    # SEKE stats
    q_min = float(np.min(q_seke))
    q_med = float(np.median(q_seke))
    q_mean = float(np.mean(q_seke))
    q_max = float(np.max(q_seke))
    pct_hi = 100.0 * float(np.mean(q_seke > 0.5))

    # Random baseline (same seed) for readable deltas
    rng = np.random.default_rng(rng_seed)
    perm_rand = np.arange(n, dtype=np.int32)
    rng.shuffle(perm_rand)
    rank_rand = np.empty(n, dtype=np.int32)
    rank_rand[perm_rand] = np.arange(n, dtype=np.int32)
    tot0, fw0, bw0 = compute_forward_backward(edges_indexed, rank_rand)

    with open(out_log_path, "w") as logf:
        logf.write("[meta] method=seke_rocketcrane_cycle_ablation\n")
        logf.write(f"[meta] input={dimacs_path}\n")
        logf.write(f"[meta] seed={rng_seed}\n")
        logf.write(f"[meta] n={n} m={m}\n")
        logf.write(f"[meta] time_limit_s={time_limit_s}\n")
        logf.write(f"[meta] seke_k={seke_k} seke_tau={tau_used:.6g} lambda_seke={lambda_seke}\n")
        logf.write(f"[seke] q_min={q_min:.6f} q_med={q_med:.6f} q_mean={q_mean:.6f} q_max={q_max:.6f} pct_q_gt_0p5={pct_hi:.4f}\n")
        logf.write(f"[random_baseline] fw={fw0:.6f} bw={bw0:.6f} ratio={fw0/tot0:.12f}\n")
        logf.flush()

        # Rocket
        perm_r, rank_r, _ = rocket_optimize_seke_cycleaware(
            n=n, eu=eu, ev=ev, ew=ew,
            q_seke=q_seke,
            lambda_seke=lambda_seke,
            beta=rocket_beta,
            iters=rocket_iters,
            lr=rocket_lr,
            seed=rng_seed,
            deadline=deadline,
            logf=logf,
            log_every=rocket_log_every
        )
        tot1, fw1, bw1 = compute_forward_backward(edges_indexed, rank_r)
        logf.write(f"[after_rocket] fw={fw1:.6f} bw={bw1:.6f} ratio={fw1/tot1:.12f}\n")
        logf.flush()

        # Crane
        perm_c, rank_c, _ = crane_refine(
            perm_init=perm_r,
            eu=eu, ev=ev, ew=ew,
            out_nbrs=out_nbrs, out_w=out_w,
            in_nbrs=in_nbrs, in_w=in_w,
            seed=rng_seed + 1,
            sa_steps=crane_sa_steps,
            T0=crane_T0,
            alpha=crane_alpha,
            mc_steps=crane_mc_steps,
            deadline=deadline,
            logf=logf,
            sa_log_every=crane_sa_log_every,
            mc_log_every=crane_mc_log_every,
        )
        tot2, fw2, bw2 = compute_forward_backward(edges_indexed, rank_c)
        logf.write(f"[after_crane] fw={fw2:.6f} bw={bw2:.6f} ratio={fw2/tot2:.12f}\n")
        logf.flush()

        # Final guarantee: output better of after_rocket vs after_crane
        if fw2 >= fw1 - 1e-9:
            final_rank = rank_c
            final_fw, final_bw, final_tot = fw2, bw2, tot2
            chosen = "rocket(+seke)+crane"
        else:
            final_rank = rank_r
            final_fw, final_bw, final_tot = fw1, bw1, tot1
            chosen = "rocket(+seke)_only"

        write_ranking_csv_nodeid_order(out_csv_path, index_to_node, final_rank)

        elapsed = time.time() - t0
        logf.write(f"[final] chosen={chosen}\n")
        logf.write(f"[final] fw={final_fw:.6f} bw={final_bw:.6f} ratio={final_fw/final_tot:.12f}\n")
        logf.write(f"[final] out_csv={out_csv_path}\n")
        logf.write(f"[final] elapsed_s={elapsed:.3f}\n")
        logf.flush()

    if print_compact:
        # Single compact block to console (no iterative spam)
        print(
            f"seed={rng_seed}  lambda={lambda_seke}  "
            f"after_rocket_ratio={ratio_percent(fw1, tot1):.6f}%  "
            f"after_crane_ratio={ratio_percent(fw2, tot2):.6f}%  "
            f"final_ratio={ratio_percent(final_fw, final_tot):.6f}%  "
            f"elapsed_s={elapsed:.1f}"
        )

    return {
        "seed": int(rng_seed),
        "lambda": float(lambda_seke),
        "n": int(n),
        "m": int(m),
        "total_w": float(final_tot),

        "rand_fw": float(fw0),
        "rand_bw": float(bw0),

        "after_rocket_fw": float(fw1),
        "after_rocket_bw": float(bw1),

        "after_crane_fw": float(fw2),
        "after_crane_bw": float(bw2),

        "final_fw": float(final_fw),
        "final_bw": float(final_bw),
        "chosen": chosen,

        "elapsed_s": float(elapsed),
        "out_csv": out_csv_path,
        "out_log": out_log_path,

        "seke_tau": float(tau_used),
        "seke_q_min": q_min,
        "seke_q_med": q_med,
        "seke_q_mean": q_mean,
        "seke_q_max": q_max,
        "seke_q_pct_gt_0p5": pct_hi,
    }


# ============================================================
# 8) Main: paired multi-seed ablation + aggregate assessment
# ============================================================

if __name__ == "__main__":
    # ----- Input file (fixed, no CLI)
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    # ----- Output directory
    # Prefer Desktop if present; else current dir
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    out_root = desktop if os.path.isdir(desktop) else "."
    out_dir = safe_mkdir(os.path.join(out_root, "seke_ablation_outputs"))

    # ----- Method name (no timestamps)
    base = os.path.basename(edge_file)
    base = base[:-2] if base.endswith(".d") else base
    method = "seke_rocketcrane_cycle"

    # ----- Shared hyperparameters (identical across baseline & SEKE)
    TIME_LIMIT_S = None  # e.g., 3600.0 if you want a strict wall-clock limit

    SEKE_K = 20.0
    SEKE_TAU = None
    LAM_SEKE = 0.5

    ROCKET_ITERS = 300
    ROCKET_BETA = 1.0
    ROCKET_LR = 0.05

    CRANE_SA_STEPS = 25000
    CRANE_T0 = 1.0
    CRANE_ALPHA = 0.95
    CRANE_MC_STEPS = 25000

    # ----- Stronger assessment: multiple seeds (paired)
    # Keep small by default; increase if you want stronger evidence
    SEEDS = [1, 2, 3, 4, 5]

    # ----- Console verbosity
    PRINT_PER_RUN = True  # still compact (one line per run)
    PRINT_HEADER = True

    if PRINT_HEADER:
        print("\n================= SEKE ABLATION (multi-seed, paired) =================")
        print(f"Input:   {edge_file}")
        print(f"Out dir: {out_dir}")
        print(f"Seeds:   {SEEDS}")
        print(f"Budgets: rocket_iters={ROCKET_ITERS}, SA={CRANE_SA_STEPS}, MC={CRANE_MC_STEPS}, time_limit={TIME_LIMIT_S}")
        print(f"Lambdas: baseline=0.0, seke={LAM_SEKE}")
        print("-----------------------------------------------------------------------")

    all_runs = []

    # Run baseline and SEKE for each seed (paired)
    for seed in SEEDS:
        for lam in [0.0, float(LAM_SEKE)]:
            lam_tag = f"lam{fmt_lam(lam)}"
            out_csv = os.path.join(out_dir, f"{base}_{method}_seed{seed}_{lam_tag}_ranking.csv")
            out_log = os.path.join(out_dir, f"{base}_{method}_seed{seed}_{lam_tag}.log")

            res = run_one_lambda(
                dimacs_path=edge_file,
                out_csv_path=out_csv,
                out_log_path=out_log,
                rng_seed=seed,
                time_limit_s=TIME_LIMIT_S,

                seke_k=SEKE_K,
                seke_tau=SEKE_TAU,
                lambda_seke=lam,

                rocket_iters=ROCKET_ITERS,
                rocket_beta=ROCKET_BETA,
                rocket_lr=ROCKET_LR,
                rocket_log_every=50,     # log file only

                crane_sa_steps=CRANE_SA_STEPS,
                crane_T0=CRANE_T0,
                crane_alpha=CRANE_ALPHA,
                crane_mc_steps=CRANE_MC_STEPS,
                crane_sa_log_every=5000, # log file only
                crane_mc_log_every=5000, # log file only

                print_compact=PRINT_PER_RUN
            )
            all_runs.append(res)

    df = pd.DataFrame(all_runs)
    summary_csv = os.path.join(out_dir, f"{base}_{method}_ALL_RUNS.csv")
    df.to_csv(summary_csv, index=False)

    # Paired deltas per seed: (SEKE - baseline)
    def pick(df_in, lam):
        return df_in[df_in["lambda"] == float(lam)].copy()

    df0 = pick(df, 0.0).set_index("seed")
    df1 = pick(df, float(LAM_SEKE)).set_index("seed")

    seeds_common = sorted(set(df0.index).intersection(set(df1.index)))
    rows = []
    for s in seeds_common:
        r0 = df0.loc[s]
        r1 = df1.loc[s]
        tot = float(r0["total_w"])  # same graph, should match
        d_rocket_fw = float(r1["after_rocket_fw"]) - float(r0["after_rocket_fw"])
        d_crane_fw  = float(r1["after_crane_fw"])  - float(r0["after_crane_fw"])
        d_final_fw  = float(r1["final_fw"])        - float(r0["final_fw"])

        d_rocket_ratio = ratio_percent(float(r1["after_rocket_fw"]), tot) - ratio_percent(float(r0["after_rocket_fw"]), tot)
        d_crane_ratio  = ratio_percent(float(r1["after_crane_fw"]),  tot) - ratio_percent(float(r0["after_crane_fw"]),  tot)
        d_final_ratio  = ratio_percent(float(r1["final_fw"]),        tot) - ratio_percent(float(r0["final_fw"]),        tot)

        # “washout” / retention: how much of Rocket delta survives after Crane
        # (positive means Crane keeps/improves the SEKE advantage)
        retention_fw = d_crane_fw - d_rocket_fw

        rows.append({
            "seed": int(s),
            "delta_after_rocket_fw": d_rocket_fw,
            "delta_after_crane_fw": d_crane_fw,
            "delta_final_fw": d_final_fw,
            "delta_after_rocket_ratio_pp": d_rocket_ratio,  # percentage points
            "delta_after_crane_ratio_pp": d_crane_ratio,
            "delta_final_ratio_pp": d_final_ratio,
            "retention_fw_(after_crane_minus_after_rocket_delta)": retention_fw,
            "baseline_final_ratio_%": ratio_percent(float(r0["final_fw"]), tot),
            "seke_final_ratio_%": ratio_percent(float(r1["final_fw"]), tot),
            "baseline_csv": str(r0["out_csv"]),
            "seke_csv": str(r1["out_csv"]),
        })

    df_delta = pd.DataFrame(rows).sort_values("seed")
    delta_csv = os.path.join(out_dir, f"{base}_{method}_DELTAS_PAIRED.csv")
    df_delta.to_csv(delta_csv, index=False)

    # Aggregate stats
    def agg_stats(col):
        vals = df_delta[col].to_numpy(dtype=float)
        return float(np.mean(vals)), float(np.std(vals, ddof=1)) if len(vals) >= 2 else 0.0

    mean_dR_fw, std_dR_fw = agg_stats("delta_after_rocket_fw")
    mean_dC_fw, std_dC_fw = agg_stats("delta_after_crane_fw")
    mean_dF_fw, std_dF_fw = agg_stats("delta_final_fw")

    mean_dR_pp, std_dR_pp = agg_stats("delta_after_rocket_ratio_pp")
    mean_dC_pp, std_dC_pp = agg_stats("delta_after_crane_ratio_pp")
    mean_dF_pp, std_dF_pp = agg_stats("delta_final_ratio_pp")

    win_rate_final = float(np.mean(df_delta["delta_final_fw"].to_numpy(dtype=float) > 0.0)) * 100.0

    compare_log = os.path.join(out_dir, f"{base}_{method}_COMPARE.log")
    with open(compare_log, "w") as f:
        f.write("[compare] method=seke_rocketcrane_cycle_ablation (multi-seed paired)\n")
        f.write(f"[compare] input={edge_file}\n")
        f.write(f"[compare] out_dir={out_dir}\n")
        f.write(f"[compare] seeds={SEEDS}\n")
        f.write(f"[compare] budgets rocket_iters={ROCKET_ITERS} SA={CRANE_SA_STEPS} MC={CRANE_MC_STEPS} time_limit_s={TIME_LIMIT_S}\n")
        f.write(f"[compare] lambda_seke={LAM_SEKE}\n\n")

        f.write(f"[files] all_runs_csv={summary_csv}\n")
        f.write(f"[files] deltas_csv={delta_csv}\n\n")

        f.write("[aggregate deltas: SEKE - baseline]\n")
        f.write(f"  after_rocket: mean ΔFW={mean_dR_fw:+.6f}  std={std_dR_fw:.6f}   mean Δratio_pp={mean_dR_pp:+.6f}  std={std_dR_pp:.6f}\n")
        f.write(f"  after_crane : mean ΔFW={mean_dC_fw:+.6f}  std={std_dC_fw:.6f}   mean Δratio_pp={mean_dC_pp:+.6f}  std={std_dC_pp:.6f}\n")
        f.write(f"  final       : mean ΔFW={mean_dF_fw:+.6f}  std={std_dF_fw:.6f}   mean Δratio_pp={mean_dF_pp:+.6f}  std={std_dF_pp:.6f}\n")
        f.write(f"  final win-rate (ΔFW>0): {win_rate_final:.2f}%\n")

    # Final concise console summary
    print("\n================= PAIRED ABLATION SUMMARY =================")
    print(f"Seeds paired: {seeds_common}")
    print(f"Output files:")
    print(f"  ALL runs:    {summary_csv}")
    print(f"  DELTAS:      {delta_csv}")
    print(f"  Compare log: {compare_log}")
    print("")
    print("Aggregate (SEKE - baseline):")
    print(f"  after Rocket: mean ΔFW={mean_dR_fw:+.6f}  (std={std_dR_fw:.6f})   mean Δratio={mean_dR_pp:+.6f} pp (std={std_dR_pp:.6f})")
    print(f"  after Crane : mean ΔFW={mean_dC_fw:+.6f}  (std={std_dC_fw:.6f})   mean Δratio={mean_dC_pp:+.6f} pp (std={std_dC_pp:.6f})")
    print(f"  final       : mean ΔFW={mean_dF_fw:+.6f}  (std={std_dF_fw:.6f})   mean Δratio={mean_dF_pp:+.6f} pp (std={std_dF_pp:.6f})")
    print(f"  final win-rate (ΔFW>0): {win_rate_final:.2f}%")
    print("===========================================================\n")



================= SEKE ABLATION (multi-seed, paired) =================
Input:   /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
Out dir: ./seke_ablation_outputs
Seeds:   [1, 2, 3, 4, 5]
Budgets: rocket_iters=300, SA=25000, MC=25000, time_limit=None
Lambdas: baseline=0.0, seke=0.5
-----------------------------------------------------------------------
seed=1  lambda=0.0  after_rocket_ratio=79.433019%  after_crane_ratio=80.066659%  final_ratio=80.066659%  elapsed_s=78.8
seed=1  lambda=0.5  after_rocket_ratio=79.573131%  after_crane_ratio=80.193226%  final_ratio=80.193226%  elapsed_s=75.4
seed=2  lambda=0.0  after_rocket_ratio=79.471846%  after_crane_ratio=80.108508%  final_ratio=80.108508%  elapsed_s=77.0
seed=2  lambda=0.5  after_rocket_ratio=79.585102%  after_crane_ratio=80.203044%  final_ratio=80.203044%  elapsed_s=76.0
seed=3  lambda=0.0  after_rocket_ratio=79.482895%  after_crane_ratio=80.115750%  final_ratio=80.115750%  elapsed_s=77.8
seed=3  lambda=0.5  after_rocket

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
NEXT STEP (per advisor): implement alternative SEKE-stability losses + lambda sweep + Rocket timing.

What you get in this script:
  ✅ Same DIMACS (.d) reader w/ aggregation (deterministic)
  ✅ Same output ranking CSV schema: ["Node ID", "Order"]
  ✅ Rocket supports:
        - edge-local SEKE term (your current q_e regularizer)
        - direct vertex-level stability losses:
            * hinge-squared (Option A)
            * softplus barrier (Option B)
        - optional per-vertex normalization (recommended)
  ✅ Sweep over lambda values (paired multi-seed like your ablation)
  ✅ Logs include Rocket timing and breakdown (confirm Rocket is fast)

Run: just edit __main__ inputs and execute.
"""

import os
import time
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd


# ============================================================
# 0) Helpers
# ============================================================

def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

def ratio_percent(fw, total_w):
    return (100.0 * fw / total_w) if total_w > 0 else 0.0

def fmt_lam(lam: float):
    s = f"{lam:.6g}"
    return s.replace(".", "p").replace("-", "m")

def sigmoid(x):
    # stable enough for float32 ranges used here; keep simple
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    # numerically stable softplus
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)

def write_ranking_csv_nodeid_order(path, index_to_node, rank_arr):
    n = len(rank_arr)
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank_arr[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(path, index=False)


# ============================================================
# 1) DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Forward/backward weight given a rank array
# ============================================================

def compute_forward_backward(edges_indexed, rank_arr):
    total_w = 0.0
    fw = 0.0
    for u, v, w in edges_indexed:
        total_w += w
        if rank_arr[u] < rank_arr[v]:
            fw += w
    bw = total_w - fw
    return total_w, fw, bw


# ============================================================
# 3) SEKE suspiciousness q_e (static)
# ============================================================

def build_seke_edge_suspiciousness(n: int, eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
                                  k: float = 20.0, tau=None, eps: float = 1e-12):
    Win = np.zeros(n, dtype=np.float64)
    Wout = np.zeros(n, dtype=np.float64)
    np.add.at(Wout, eu, ew.astype(np.float64))
    np.add.at(Win,  ev, ew.astype(np.float64))

    denom = Win[eu].astype(np.float64) + Wout[ev].astype(np.float64) + eps
    s = (ew.astype(np.float64) / denom)

    if tau is None:
        tau = float(np.median(s))

    x = float(k) * (s - float(tau))
    q = 1.0 / (1.0 + np.exp(-x))
    return q.astype(np.float32), float(tau)


# ============================================================
# 4) Crane (unchanged): TopoShuffle init + SA + greedy swaps
# ============================================================

def build_inc_out_lists(n, eu, ev, ew):
    out_nbrs = [[] for _ in range(n)]
    out_w = [[] for _ in range(n)]
    in_nbrs = [[] for _ in range(n)]
    in_w = [[] for _ in range(n)]
    for u, v, w in zip(eu.tolist(), ev.tolist(), ew.tolist()):
        out_nbrs[u].append(v); out_w[u].append(w)
        in_nbrs[v].append(u);  in_w[v].append(w)
    out_nbrs = [np.array(x, dtype=np.int32) for x in out_nbrs]
    out_w = [np.array(x, dtype=np.float32) for x in out_w]
    in_nbrs = [np.array(x, dtype=np.int32) for x in in_nbrs]
    in_w = [np.array(x, dtype=np.float32) for x in in_w]
    return out_nbrs, out_w, in_nbrs, in_w

def toposhuffle_init(n, perm, eu, ev, seed=1):
    rng = random.Random(seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    out = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in zip(eu.tolist(), ev.tolist()):
        if rank[u] < rank[v]:
            out[u].append(v)
            indeg[v] += 1

    zeros = [i for i in range(n) if indeg[i] == 0]
    rng.shuffle(zeros)

    order = []
    while zeros:
        u = zeros.pop()
        order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                zeros.append(v)
                j = rng.randrange(len(zeros))
                zeros[-1], zeros[j] = zeros[j], zeros[-1]

    if len(order) != n:
        seen = set(order)
        for i in range(n):
            if i not in seen:
                order.append(i)

    return np.array(order, dtype=np.int32)

def delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w):
    ra = int(rank[a]); rb = int(rank[b])
    if ra == rb:
        return 0.0

    def r_after(x):
        if x == a: return rb
        if x == b: return ra
        return int(rank[x])

    delta = 0.0

    for u in (a, b):
        ru0 = int(rank[u]); ru1 = r_after(u)
        nbrs = out_nbrs[u]; ws = out_w[u]
        if nbrs.size:
            rv0 = rank[nbrs]
            rv1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[ru0 < rv0]))
            after  = float(np.sum(ws[ru1 < rv1]))
            delta += (after - before)

    for u in (a, b):
        ru0 = int(rank[u]); ru1 = r_after(u)
        nbrs = in_nbrs[u]; ws = in_w[u]
        if nbrs.size:
            rx0 = rank[nbrs]
            rx1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[rx0 < ru0]))
            after  = float(np.sum(ws[rx1 < ru1]))
            delta += (after - before)

    def direct_w(u, v):
        nbrs = out_nbrs[u]; ws = out_w[u]
        if nbrs.size == 0: return 0.0
        idx = np.where(nbrs == v)[0]
        if idx.size == 0: return 0.0
        return float(np.sum(ws[idx]))

    wab = direct_w(a, b); wba = direct_w(b, a)

    if wab != 0.0:
        before = wab if ra < rb else 0.0
        after  = wab if rb < ra else 0.0
        delta -= (after - before)

    if wba != 0.0:
        before = wba if rb < ra else 0.0
        after  = wba if ra < rb else 0.0
        delta -= (after - before)

    return delta

def crane_refine(
    perm_init,
    eu, ev, ew,
    out_nbrs, out_w, in_nbrs, in_w,
    seed=2,
    sa_steps=25000,
    T0=1.0,
    alpha=0.95,
    mc_steps=25000,
    deadline=None,
    logf=None,
    sa_log_every=5000,
    mc_log_every=5000,
):
    rng = random.Random(seed)
    n = len(perm_init)

    perm = toposhuffle_init(n, perm_init, eu, ev, seed=seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    best_perm = perm.copy()
    best_rank = rank.copy()
    best_fw = fw

    if logf:
        logf.write(f"[crane] after_toposhuffle_fw={best_fw:.6f}\n")
        logf.flush()

    T = float(T0)
    cur_fw = fw

    for it in range(1, sa_steps + 1):
        if deadline is not None and time.time() >= deadline:
            if logf:
                logf.write(f"[crane][sa] stop(deadline) it={it}\n")
                logf.flush()
            break

        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue

        a = int(perm[a_pos]); b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)

        accept = (d >= 0) or (T > 1e-12 and rng.random() < math.exp(d / T))
        if accept:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)

            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

        T *= alpha

        if logf and sa_log_every and (it % sa_log_every == 0 or it == 1 or it == sa_steps):
            logf.write(f"[crane][sa] it={it} cur_fw={cur_fw:.6f} best_fw={best_fw:.6f} T={T:.4g}\n")
            logf.flush()

    perm = best_perm.copy()
    rank = best_rank.copy()
    cur_fw = best_fw

    for it in range(1, mc_steps + 1):
        if deadline is not None and time.time() >= deadline:
            if logf:
                logf.write(f"[crane][mc] stop(deadline) it={it}\n")
                logf.flush()
            break

        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue

        a = int(perm[a_pos]); b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)

        if d > 0:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)

            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

        if logf and mc_log_every and (it % mc_log_every == 0 or it == 1 or it == mc_steps):
            logf.write(f"[crane][mc] it={it} cur_fw={cur_fw:.6f} best_fw={best_fw:.6f}\n")
            logf.flush()

    return best_perm, best_rank, best_fw


# ============================================================
# 5) Direct stability loss (Option A/B) + gradient
# ============================================================

def stability_loss_and_edge_dLd_d(
    n,
    eu, ev, ew,
    P,
    beta: float,
    loss_type: str,          # "none" | "hinge2" | "softplus"
    lam_stab: float,
    normalize: bool = True,
    eps_norm: float = 1e-12,
    margin: float = 0.0,
):
    """
    Returns:
      L_stab (float)
      edge_dLd_d (np.float32 array length m): derivative d( L_stab ) / d d_e, where d_e = P[v]-P[u].
    Note: this is the derivative of L_stab only (not multiplied by lam_stab).
    """

    if loss_type == "none" or lam_stab == 0.0:
        return 0.0, np.zeros_like(ew, dtype=np.float32)

    # d_e = P[v] - P[u]
    d = (P[ev] - P[eu]).astype(np.float32)
    x = float(beta) * d
    fwd = sigmoid(x).astype(np.float32)
    bwd = (1.0 - fwd).astype(np.float32)

    # per-vertex aggregates
    BackIn  = np.zeros(n, dtype=np.float64)  # sum_in w*bwd
    FwdOut  = np.zeros(n, dtype=np.float64)  # sum_out w*fwd
    BackOut = np.zeros(n, dtype=np.float64)  # sum_out w*bwd
    FwdIn   = np.zeros(n, dtype=np.float64)  # sum_in w*fwd

    w64 = ew.astype(np.float64)
    np.add.at(BackIn,  ev, w64 * bwd.astype(np.float64))
    np.add.at(FwdOut,  eu, w64 * fwd.astype(np.float64))
    np.add.at(BackOut, eu, w64 * bwd.astype(np.float64))
    np.add.at(FwdIn,   ev, w64 * fwd.astype(np.float64))

    v1 = BackIn - FwdOut
    v2 = BackOut - FwdIn

    if normalize:
        Tot = np.zeros(n, dtype=np.float64)
        np.add.at(Tot, eu, w64)  # out
        np.add.at(Tot, ev, w64)  # in
        denom = Tot + float(eps_norm)
        v1n = v1 / denom
        v2n = v2 / denom
    else:
        denom = None
        v1n, v2n = v1, v2

    # dL/d(v1) and dL/d(v2) (vertex-level)
    if loss_type == "hinge2":
        r1 = np.maximum(v1n, 0.0)
        r2 = np.maximum(v2n, 0.0)
        L = float(np.sum(r1 * r1) + np.sum(r2 * r2))
        # derivative wrt v1n is 2*r1 (when >0), similarly v2n
        dv1n = (2.0 * r1).astype(np.float64)
        dv2n = (2.0 * r2).astype(np.float64)
        # chain back if normalized
        if normalize:
            dv1 = dv1n / denom
            dv2 = dv2n / denom
        else:
            dv1 = dv1n
            dv2 = dv2n

    elif loss_type == "softplus":
        # softplus(v - margin)
        z1 = v1n - float(margin)
        z2 = v2n - float(margin)
        L = float(np.sum(softplus(z1)) + np.sum(softplus(z2)))
        # d/dz softplus(z) = sigmoid(z)
        s1 = sigmoid(z1).astype(np.float64)
        s2 = sigmoid(z2).astype(np.float64)
        if normalize:
            dv1 = (s1 / denom).astype(np.float64)
            dv2 = (s2 / denom).astype(np.float64)
        else:
            dv1 = s1
            dv2 = s2
    else:
        raise ValueError(f"Unknown stability loss_type={loss_type}")

    # Now map vertex derivatives to edge derivatives via fwd/bwd dependence.
    # v1 = BackIn - FwdOut
    #   BackIn uses bwd on edges into v => contributes +dv1[v]
    #   FwdOut uses fwd on edges out of u => contributes -dv1[u]
    # v2 = BackOut - FwdIn
    #   BackOut uses bwd on edges out of u => contributes +dv2[u]
    #   FwdIn uses fwd on edges into v => contributes -dv2[v]

    # per-vertex coefficients for the four aggregates:
    g_BackIn = dv1                      # dL/dBackIn
    g_FwdOut = -dv1                     # dL/dFwdOut
    g_BackOut = dv2                     # dL/dBackOut
    g_FwdIn = -dv2                      # dL/dFwdIn

    # df/dd = beta * fwd*(1-fwd) ; db/dd = -df/dd
    df_dd = (float(beta) * (fwd * (1.0 - fwd))).astype(np.float32)
    db_dd = (-df_dd).astype(np.float32)

    # edge-wise dL/dd
    #   BackIn term uses bwd at vertex v: g_BackIn[v] * w * db/dd
    #   FwdOut term uses fwd at vertex u: g_FwdOut[u] * w * df/dd
    #   BackOut term uses bwd at vertex u: g_BackOut[u] * w * db/dd
    #   FwdIn term uses fwd at vertex v: g_FwdIn[v] * w * df/dd
    edge_dLd_d = (
        (ew * db_dd) * (g_BackIn[ev].astype(np.float32) + g_BackOut[eu].astype(np.float32)) +
        (ew * df_dd) * (g_FwdOut[eu].astype(np.float32) + g_FwdIn[ev].astype(np.float32))
    ).astype(np.float32)

    return L, edge_dLd_d


# ============================================================
# 6) Rocket (with configurable losses) — Adam minimize Loss
#    Loss = - Σ ŵ σ(βΔ) - λ_seke Σ ŵ q σ(-βΔ) + λ_stab * L_stab(P)
# ============================================================

def rocket_optimize_with_losses(
    n,
    eu, ev, ew,
    q_seke,
    lambda_seke: float = 0.0,
    loss_type_stab: str = "none",    # "none"|"hinge2"|"softplus"
    lambda_stab: float = 0.0,
    stab_normalize: bool = True,
    stab_margin: float = 0.0,
    beta: float = 1.0,
    iters: int = 300,
    lr: float = 0.05,
    seed: int = 1,
    deadline=None,
    logf=None,
    log_every: int = 50,
):
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    # weight normalization for Rocket smooth objective
    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    lam = float(lambda_seke)
    lam_stab = float(lambda_stab)
    bet = float(beta)

    # Adam
    m = np.zeros(n, dtype=np.float32)
    v = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def perm_from_positions(Pvec):
        return np.argsort(Pvec, kind="mergesort").astype(np.int32)

    def rank_from_perm(perm):
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    # best discrete FW checkpoint
    perm0 = perm_from_positions(P)
    rank0 = rank_from_perm(perm0)
    fw_best = float(np.sum(ew[rank0[eu] < rank0[ev]]))
    perm_best = perm0.copy()
    rank_best = rank0.copy()

    t_rocket0 = time.time()

    if logf:
        logf.write(f"[rocket] init_best_fw={fw_best:.6f}  lambda_seke={lam:.6g}  stab={loss_type_stab} lambda_stab={lam_stab:.6g}\n")
        logf.flush()

    for it in range(1, iters + 1):
        if deadline is not None and time.time() >= deadline:
            if logf:
                logf.write(f"[rocket] stop(deadline) it={it}\n")
                logf.flush()
            break

        d = (P[ev] - P[eu]).astype(np.float32)          # Δ
        x = (bet * d).astype(np.float32)

        sig_f = sigmoid(x).astype(np.float32)
        sigp = (sig_f * (1.0 - sig_f)).astype(np.float32)

        # main objective gradient (same behavior as your current implementation)
        # edge_grad_u = + (1 - lam*q) * w_hat * sigp * beta
        edge_grad_u = (w_hat * (1.0 - lam * q_seke) * sigp * bet).astype(np.float32)

        # direct stability penalty (adds to Loss, so its gradient adds)
        if loss_type_stab != "none" and lam_stab != 0.0:
            L_stab, edge_dLd_d = stability_loss_and_edge_dLd_d(
                n=n, eu=eu, ev=ev, ew=ew,
                P=P, beta=bet,
                loss_type=loss_type_stab,
                lam_stab=lam_stab,
                normalize=stab_normalize,
                margin=stab_margin,
            )
            # Loss includes + lam_stab*L_stab, so dLoss/dP adds lam_stab*dL_stab/dP.
            # edge_dLd_d is d(L_stab)/dd; mapping:
            #   dLoss/dP[u] += -lam_stab * dL/dd
            #   dLoss/dP[v] += +lam_stab * dL/dd
            edge_grad_u = (edge_grad_u - lam_stab * edge_dLd_d).astype(np.float32)
        else:
            L_stab = 0.0

        # scatter to node gradients (this is dLoss/dP)
        grad = np.zeros(n, dtype=np.float32)
        np.add.at(grad, eu, +edge_grad_u)
        np.add.at(grad, ev, -edge_grad_u)

        # Adam step (minimize Loss)
        t += 1
        m = (b1 * m + (1 - b1) * grad).astype(np.float32)
        v = (b2 * v + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m / (1 - (b1 ** t))
        vhat = v / (1 - (b2 ** t))
        P = (P - float(lr) * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # discrete FW checkpoint
        perm = perm_from_positions(P)
        rank = rank_from_perm(perm)
        fw = float(np.sum(ew[rank[eu] < rank[ev]]))
        if fw > fw_best + 1e-9:
            fw_best = fw
            perm_best = perm.copy()
            rank_best = rank.copy()

        if logf and log_every and (it % log_every == 0 or it == 1 or it == iters):
            logf.write(f"[rocket] it={it} cur_fw={fw:.6f} best_fw={fw_best:.6f} L_stab={L_stab:.6f}\n")
            logf.flush()

    rocket_elapsed = time.time() - t_rocket0
    return perm_best, rank_best, fw_best, rocket_elapsed


# ============================================================
# 7) One run (seed + lambdas + loss mode)
# ============================================================

def run_one_config(
    dimacs_path: str,
    out_csv_path: str,
    out_log_path: str,
    rng_seed=1,
    time_limit_s=None,

    # SEKE q params
    seke_k=20.0,
    seke_tau=None,

    # Rocket params
    rocket_iters=300,
    rocket_beta=1.0,
    rocket_lr=0.05,
    rocket_log_every=50,

    # Loss toggles/weights
    lambda_seke=0.0,                # edge-local q term strength
    stab_loss_type="none",          # "none"|"hinge2"|"softplus"
    lambda_stab=0.0,                # stability loss strength
    stab_normalize=True,
    stab_margin=0.0,

    # Crane params
    crane_sa_steps=25000,
    crane_T0=1.0,
    crane_alpha=0.95,
    crane_mc_steps=25000,
    crane_sa_log_every=5000,
    crane_mc_log_every=5000,

    # Console
    print_compact=True,
):
    t0 = time.time()
    deadline = (t0 + float(time_limit_s)) if (time_limit_s is not None and time_limit_s > 0) else None

    t_read0 = time.time()
    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(dimacs_path)
    n = len(node_to_index)
    m = len(edges_indexed)
    t_read = time.time() - t_read0

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges_indexed):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    t_pre0 = time.time()
    out_nbrs, out_w, in_nbrs, in_w = build_inc_out_lists(n, eu, ev, ew)
    q_seke, tau_used = build_seke_edge_suspiciousness(n, eu, ev, ew, k=float(seke_k), tau=seke_tau)
    t_pre = time.time() - t_pre0

    # random baseline (for log readability)
    rng = np.random.default_rng(rng_seed)
    perm_rand = np.arange(n, dtype=np.int32)
    rng.shuffle(perm_rand)
    rank_rand = np.empty(n, dtype=np.int32)
    rank_rand[perm_rand] = np.arange(n, dtype=np.int32)
    tot0, fw0, bw0 = compute_forward_backward(edges_indexed, rank_rand)

    with open(out_log_path, "w") as logf:
        logf.write("[meta] method=rocketcrane_with_stability_losses\n")
        logf.write(f"[meta] input={dimacs_path}\n")
        logf.write(f"[meta] seed={rng_seed}\n")
        logf.write(f"[meta] n={n} m={m}\n")
        logf.write(f"[meta] time_limit_s={time_limit_s}\n")
        logf.write(f"[meta] read_s={t_read:.4f} precompute_s={t_pre:.4f}\n")
        logf.write(f"[meta] seke_k={seke_k} seke_tau={tau_used:.6g}\n")
        logf.write(f"[meta] lambda_seke={lambda_seke}  stab_loss={stab_loss_type}  lambda_stab={lambda_stab}  norm={stab_normalize} margin={stab_margin}\n")

        q_min = float(np.min(q_seke)); q_med = float(np.median(q_seke))
        q_mean = float(np.mean(q_seke)); q_max = float(np.max(q_seke))
        pct_hi = 100.0 * float(np.mean(q_seke > 0.5))
        logf.write(f"[seke] q_min={q_min:.6f} q_med={q_med:.6f} q_mean={q_mean:.6f} q_max={q_max:.6f} pct_q_gt_0p5={pct_hi:.4f}\n")
        logf.write(f"[random_baseline] fw={fw0:.6f} bw={bw0:.6f} ratio={fw0/tot0:.12f}\n")
        logf.flush()

        # Rocket
        perm_r, rank_r, fw_r, rocket_elapsed = rocket_optimize_with_losses(
            n=n, eu=eu, ev=ev, ew=ew,
            q_seke=q_seke,
            lambda_seke=float(lambda_seke),
            loss_type_stab=str(stab_loss_type),
            lambda_stab=float(lambda_stab),
            stab_normalize=bool(stab_normalize),
            stab_margin=float(stab_margin),
            beta=float(rocket_beta),
            iters=int(rocket_iters),
            lr=float(rocket_lr),
            seed=int(rng_seed),
            deadline=deadline,
            logf=logf,
            log_every=int(rocket_log_every),
        )
        tot1, fw1, bw1 = compute_forward_backward(edges_indexed, rank_r)
        logf.write(f"[after_rocket] fw={fw1:.6f} bw={bw1:.6f} ratio={fw1/tot1:.12f}\n")
        logf.write(f"[timing] rocket_s={rocket_elapsed:.4f}\n")
        logf.flush()

        # Crane
        t_cr0 = time.time()
        perm_c, rank_c, fw_c = crane_refine(
            perm_init=perm_r,
            eu=eu, ev=ev, ew=ew,
            out_nbrs=out_nbrs, out_w=out_w,
            in_nbrs=in_nbrs, in_w=in_w,
            seed=int(rng_seed) + 1,
            sa_steps=int(crane_sa_steps),
            T0=float(crane_T0),
            alpha=float(crane_alpha),
            mc_steps=int(crane_mc_steps),
            deadline=deadline,
            logf=logf,
            sa_log_every=int(crane_sa_log_every),
            mc_log_every=int(crane_mc_log_every),
        )
        crane_elapsed = time.time() - t_cr0
        tot2, fw2, bw2 = compute_forward_backward(edges_indexed, rank_c)
        logf.write(f"[after_crane] fw={fw2:.6f} bw={bw2:.6f} ratio={fw2/tot2:.12f}\n")
        logf.write(f"[timing] crane_s={crane_elapsed:.4f}\n")
        logf.flush()

        # Final guarantee: choose better of after_rocket vs after_crane
        if fw2 >= fw1 - 1e-9:
            final_rank = rank_c
            final_fw, final_bw, final_tot = fw2, bw2, tot2
            chosen = "rocket+crane"
        else:
            final_rank = rank_r
            final_fw, final_bw, final_tot = fw1, bw1, tot1
            chosen = "rocket_only"

        write_ranking_csv_nodeid_order(out_csv_path, index_to_node, final_rank)

        elapsed = time.time() - t0
        logf.write(f"[final] chosen={chosen}\n")
        logf.write(f"[final] fw={final_fw:.6f} bw={final_bw:.6f} ratio={final_fw/final_tot:.12f}\n")
        logf.write(f"[final] out_csv={out_csv_path}\n")
        logf.write(f"[final] elapsed_s={elapsed:.3f}\n")
        logf.flush()

    if print_compact:
        print(
            f"seed={rng_seed}  "
            f"lamSEKE={lambda_seke} lamSTAB={lambda_stab} stab={stab_loss_type}  "
            f"after_rocket={ratio_percent(fw1, tot1):.6f}%  "
            f"after_crane={ratio_percent(fw2, tot2):.6f}%  "
            f"final={ratio_percent(final_fw, final_tot):.6f}%  "
            f"rocket_s={rocket_elapsed:.2f} crane_s={crane_elapsed:.2f} total_s={elapsed:.2f}"
        )

    return {
        "seed": int(rng_seed),
        "n": int(n),
        "m": int(m),
        "total_w": float(final_tot),

        "lambda_seke": float(lambda_seke),
        "stab_loss": str(stab_loss_type),
        "lambda_stab": float(lambda_stab),
        "stab_normalize": bool(stab_normalize),
        "stab_margin": float(stab_margin),

        "after_rocket_fw": float(fw1),
        "after_crane_fw": float(fw2),
        "final_fw": float(final_fw),
        "final_bw": float(final_bw),
        "chosen": str(chosen),

        "ratio_after_rocket_%": ratio_percent(float(fw1), float(final_tot)),
        "ratio_after_crane_%": ratio_percent(float(fw2), float(final_tot)),
        "ratio_final_%": ratio_percent(float(final_fw), float(final_tot)),

        "t_read_s": float(t_read),
        "t_precompute_s": float(t_pre),
        "t_rocket_s": float(rocket_elapsed),
        "t_crane_s": float(crane_elapsed),
        "t_total_s": float(elapsed),

        "out_csv": out_csv_path,
        "out_log": out_log_path,
    }


# ============================================================
# 8) Main: multi-seed sweep over lambda for different loss modes
# ============================================================

if __name__ == "__main__":
    # ------------------------------
    # INPUT
    # ------------------------------
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    # ------------------------------
    # OUTPUT DIR
    # ------------------------------
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    out_root = desktop if os.path.isdir(desktop) else "."
    out_dir = safe_mkdir(os.path.join(out_root, "seke_stability_sweep_outputs"))

    base = os.path.basename(edge_file)
    base = base[:-2] if base.endswith(".d") else base

    # ------------------------------
    # SHARED BUDGETS
    # ------------------------------
    TIME_LIMIT_S = None

    SEEDS = [1, 2, 3, 4, 5]

    ROCKET_ITERS = 300
    ROCKET_BETA = 1.0
    ROCKET_LR = 0.05

    CRANE_SA_STEPS = 25000
    CRANE_T0 = 1.0
    CRANE_ALPHA = 0.95
    CRANE_MC_STEPS = 25000

    # ------------------------------
    # SWEEP SETTINGS
    # ------------------------------
    # Advisor asked: sweep lambda. We'll sweep both:
    #   - lambda_seke (edge-local q)
    #   - lambda_stab (direct stability loss)
    # You can set one list to [0] to disable that dimension.

    LAM_SEKE_LIST = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
    LAM_STAB_LIST = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]  # usually enough

    # Loss modes to try (your advisor’s “alternative losses”)
    STAB_LOSSES = [
        ("none", False),      # baseline (no direct stability)
        ("hinge2", True),     # Option A normalized
        ("softplus", True),   # Option B normalized (recommended first)
    ]
    STAB_MARGIN = 0.0        # set >0 for slack if needed

    PRINT_PER_RUN = True
    print("\n================= STABILITY LOSS + LAMBDA SWEEP =================")
    print(f"Input:   {edge_file}")
    print(f"Out dir: {out_dir}")
    print(f"Seeds:   {SEEDS}")
    print(f"Budgets: rocket_iters={ROCKET_ITERS}, SA={CRANE_SA_STEPS}, MC={CRANE_MC_STEPS}, time_limit={TIME_LIMIT_S}")
    print(f"Lambda SEKE: {LAM_SEKE_LIST}")
    print(f"Lambda STAB: {LAM_STAB_LIST}")
    print("------------------------------------------------------------------")

    all_rows = []

    for stab_loss, stab_norm in STAB_LOSSES:
        for lam_seke in LAM_SEKE_LIST:
            for lam_stab in LAM_STAB_LIST:
                # If stab_loss is none, force lam_stab=0 for cleanliness
                if stab_loss == "none" and lam_stab != 0.0:
                    continue

                tag = f"stab{stab_loss}_norm{int(stab_norm)}_lamSEKE{fmt_lam(lam_seke)}_lamSTAB{fmt_lam(lam_stab)}"

                for seed in SEEDS:
                    out_csv = os.path.join(out_dir, f"{base}_Seke_{tag}_seed{seed}_ranking.csv")
                    out_log = os.path.join(out_dir, f"{base}_Seke_{tag}_seed{seed}.log")

                    res = run_one_config(
                        dimacs_path=edge_file,
                        out_csv_path=out_csv,
                        out_log_path=out_log,
                        rng_seed=seed,
                        time_limit_s=TIME_LIMIT_S,

                        seke_k=20.0,
                        seke_tau=None,

                        rocket_iters=ROCKET_ITERS,
                        rocket_beta=ROCKET_BETA,
                        rocket_lr=ROCKET_LR,
                        rocket_log_every=50,

                        lambda_seke=lam_seke,
                        stab_loss_type=stab_loss,
                        lambda_stab=lam_stab,
                        stab_normalize=stab_norm,
                        stab_margin=STAB_MARGIN,

                        crane_sa_steps=CRANE_SA_STEPS,
                        crane_T0=CRANE_T0,
                        crane_alpha=CRANE_ALPHA,
                        crane_mc_steps=CRANE_MC_STEPS,
                        crane_sa_log_every=5000,
                        crane_mc_log_every=5000,

                        print_compact=PRINT_PER_RUN
                    )
                    res["tag"] = tag
                    all_rows.append(res)

    df = pd.DataFrame(all_rows)
    all_csv = os.path.join(out_dir, f"{base}_Seke_STABILITY_SWEEP_ALL_RUNS.csv")
    df.to_csv(all_csv, index=False)

    # Aggregate summary by (stab_loss, norm, lam_seke, lam_stab)
    grp_cols = ["stab_loss", "stab_normalize", "lambda_seke", "lambda_stab"]
    agg = df.groupby(grp_cols).agg(
        mean_final_ratio=("ratio_final_%", "mean"),
        std_final_ratio=("ratio_final_%", "std"),
        mean_rocket_s=("t_rocket_s", "mean"),
        mean_crane_s=("t_crane_s", "mean"),
        mean_total_s=("t_total_s", "mean"),
        win_rate_vs_baseline=("ratio_final_%", lambda x: float(np.mean(x > 0.0))*100.0),  # placeholder (see below)
        runs=("ratio_final_%", "count"),
    ).reset_index()

    # Replace placeholder win-rate with win-rate vs the true baseline per seed:
    # baseline defined as (stab_loss="none", lam_seke=0, lam_stab=0) for same seed.
    baseline = df[(df["stab_loss"] == "none") &
                  (df["lambda_seke"] == 0.0) &
                  (df["lambda_stab"] == 0.0)][["seed", "ratio_final_%"]].set_index("seed")["ratio_final_%"]

    def win_rate_vs_baseline_for_group(sub):
        wins = 0
        total = 0
        for _, r in sub.iterrows():
            s = int(r["seed"])
            if s in baseline.index:
                total += 1
                if float(r["ratio_final_%"]) > float(baseline.loc[s]) + 1e-12:
                    wins += 1
        return 100.0 * wins / total if total else 0.0

    # compute proper win-rate per group
    win_rates = []
    for keys, sub in df.groupby(grp_cols):
        win_rates.append((*keys, win_rate_vs_baseline_for_group(sub)))

    win_df = pd.DataFrame(win_rates, columns=grp_cols + ["win_rate_vs_baseline_%"])
    agg = agg.drop(columns=["win_rate_vs_baseline"]).merge(win_df, on=grp_cols, how="left")

    agg_csv = os.path.join(out_dir, f"{base}_Seke_STABILITY_SWEEP_AGG.csv")
    agg.to_csv(agg_csv, index=False)

    print("\n================= SWEEP DONE =================")
    print(f"ALL runs CSV: {all_csv}")
    print(f"AGG summary:  {agg_csv}")
    print("Tip: sort AGG by mean_final_ratio desc, then check mean_rocket_s to answer 'Rocket is fast'.")
    print("=============================================\n")



================= STABILITY LOSS + LAMBDA SWEEP =================
Input:   /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
Out dir: ./seke_stability_sweep_outputs
Seeds:   [1, 2, 3, 4, 5]
Budgets: rocket_iters=300, SA=25000, MC=25000, time_limit=None
Lambda SEKE: [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
Lambda STAB: [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
------------------------------------------------------------------
seed=1  lamSEKE=0.0 lamSTAB=0.0 stab=none  after_rocket=79.433019%  after_crane=80.066659%  final=80.066659%  rocket_s=48.40 crane_s=7.27 total_s=79.58
seed=2  lamSEKE=0.0 lamSTAB=0.0 stab=none  after_rocket=79.471846%  after_crane=80.108508%  final=80.108508%  rocket_s=48.74 crane_s=7.07 total_s=77.51
seed=3  lamSEKE=0.0 lamSTAB=0.0 stab=none  after_rocket=79.482895%  after_crane=80.115750%  final=80.115750%  rocket_s=47.53 crane_s=7.03 total_s=73.70
seed=4  lamSEKE=0.0 lamSTAB=0.0 stab=none  after_rocket=79.465387%  after_crane=80.097736%  final=80.097736%  rocket

KeyboardInterrupt: 

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FAST SWEEP VERSION (much faster)

What makes it faster:
  ✅ Preload graph ONCE (same as your current fast script)
  ✅ FAR fewer runs:
      - Stage 1: find best λ_SEKE using stab=none  (seeds small, coarse lambdas)
      - Stage 2: fix λ_SEKE=best and sweep ONLY λ_STAB for hinge2 + softplus
  ✅ Optional "FAST MODE" budgets:
      - fewer Rocket iterations
      - fewer Crane steps
  ✅ Still prints only minimal progress and a final summary
  ✅ Outputs next to the .d file, in seke_stability_fast_outputs/

Default design:
  - Stage1 uses 3 seeds and coarse λ_SEKE list
  - Stage2 uses same 3 seeds and λ_STAB list
Then you can optionally re-run the top 1–2 configs with 5 seeds + full budgets.

IMPORTANT: This script keeps your SAME implementation for Rocket/Crane/losses.
"""

import os
import time
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd


# ============================================================
# 0) Helpers
# ============================================================

def ratio_percent(fw, total_w):
    return (100.0 * fw / total_w) if total_w > 0 else 0.0

def fmt_lam(lam: float):
    s = f"{lam:.6g}"
    return s.replace(".", "p").replace("-", "m")

def safe_sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)

def write_ranking_csv_nodeid_order(path, index_to_node, rank_arr):
    n = len(rank_arr)
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank_arr[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(path, index=False)


# ============================================================
# 1) DIMACS reader (aggregates parallel arcs) -> deterministic
# ============================================================

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Forward/backward weight given a rank array
# ============================================================

def compute_forward_backward(edges_indexed, rank_arr):
    total_w = 0.0
    fw = 0.0
    for u, v, w in edges_indexed:
        total_w += w
        if rank_arr[u] < rank_arr[v]:
            fw += w
    bw = total_w - fw
    return total_w, fw, bw


# ============================================================
# 3) SEKE suspiciousness q_e (static)
# ============================================================

def build_seke_edge_suspiciousness(n: int, eu: np.ndarray, ev: np.ndarray, ew: np.ndarray,
                                  k: float = 20.0, tau=None, eps: float = 1e-12):
    Win = np.zeros(n, dtype=np.float64)
    Wout = np.zeros(n, dtype=np.float64)
    np.add.at(Wout, eu, ew.astype(np.float64))
    np.add.at(Win,  ev, ew.astype(np.float64))

    denom = Win[eu].astype(np.float64) + Wout[ev].astype(np.float64) + eps
    s = (ew.astype(np.float64) / denom)

    if tau is None:
        tau = float(np.median(s))

    x = float(k) * (s - float(tau))
    q = 1.0 / (1.0 + np.exp(-x))
    return q.astype(np.float32), float(tau)


# ============================================================
# 4) Crane (same as your current)
# ============================================================

def build_inc_out_lists(n, eu, ev, ew):
    out_nbrs = [[] for _ in range(n)]
    out_w = [[] for _ in range(n)]
    in_nbrs = [[] for _ in range(n)]
    in_w = [[] for _ in range(n)]
    for u, v, w in zip(eu.tolist(), ev.tolist(), ew.tolist()):
        out_nbrs[u].append(v); out_w[u].append(w)
        in_nbrs[v].append(u);  in_w[v].append(w)
    out_nbrs = [np.array(x, dtype=np.int32) for x in out_nbrs]
    out_w = [np.array(x, dtype=np.float32) for x in out_w]
    in_nbrs = [np.array(x, dtype=np.int32) for x in in_nbrs]
    in_w = [np.array(x, dtype=np.float32) for x in in_w]
    return out_nbrs, out_w, in_nbrs, in_w

def toposhuffle_init(n, perm, eu, ev, seed=1):
    rng = random.Random(seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    out = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in zip(eu.tolist(), ev.tolist()):
        if rank[u] < rank[v]:
            out[u].append(v)
            indeg[v] += 1

    zeros = [i for i in range(n) if indeg[i] == 0]
    rng.shuffle(zeros)

    order = []
    while zeros:
        u = zeros.pop()
        order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                zeros.append(v)
                j = rng.randrange(len(zeros))
                zeros[-1], zeros[j] = zeros[j], zeros[-1]

    if len(order) != n:
        seen = set(order)
        for i in range(n):
            if i not in seen:
                order.append(i)

    return np.array(order, dtype=np.int32)

def delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w):
    ra = int(rank[a]); rb = int(rank[b])
    if ra == rb:
        return 0.0

    def r_after(x):
        if x == a: return rb
        if x == b: return ra
        return int(rank[x])

    delta = 0.0

    for u in (a, b):
        ru0 = int(rank[u]); ru1 = r_after(u)
        nbrs = out_nbrs[u]; ws = out_w[u]
        if nbrs.size:
            rv0 = rank[nbrs]
            rv1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[ru0 < rv0]))
            after  = float(np.sum(ws[ru1 < rv1]))
            delta += (after - before)

    for u in (a, b):
        ru0 = int(rank[u]); ru1 = r_after(u)
        nbrs = in_nbrs[u]; ws = in_w[u]
        if nbrs.size:
            rx0 = rank[nbrs]
            rx1 = np.array([r_after(int(x)) for x in nbrs], dtype=np.int32)
            before = float(np.sum(ws[rx0 < ru0]))
            after  = float(np.sum(ws[rx1 < ru1]))
            delta += (after - before)

    def direct_w(u, v):
        nbrs = out_nbrs[u]; ws = out_w[u]
        if nbrs.size == 0: return 0.0
        idx = np.where(nbrs == v)[0]
        if idx.size == 0: return 0.0
        return float(np.sum(ws[idx]))

    wab = direct_w(a, b); wba = direct_w(b, a)

    if wab != 0.0:
        before = wab if ra < rb else 0.0
        after  = wab if rb < ra else 0.0
        delta -= (after - before)

    if wba != 0.0:
        before = wba if rb < ra else 0.0
        after  = wba if ra < rb else 0.0
        delta -= (after - before)

    return delta

def crane_refine(
    perm_init,
    eu, ev, ew,
    out_nbrs, out_w, in_nbrs, in_w,
    seed=2,
    sa_steps=8000,          # DEFAULT smaller for speed
    T0=1.0,
    alpha=0.98,             # slower cooling to keep SA meaningful even with fewer steps
    mc_steps=8000,          # DEFAULT smaller for speed
    deadline=None,
):
    rng = random.Random(seed)
    n = len(perm_init)

    perm = toposhuffle_init(n, perm_init, eu, ev, seed=seed)
    rank = np.empty(n, dtype=np.int32)
    rank[perm] = np.arange(n, dtype=np.int32)

    fw = float(np.sum(ew[rank[eu] < rank[ev]]))
    best_perm = perm.copy()
    best_rank = rank.copy()
    best_fw = fw

    # SA
    T = float(T0)
    cur_fw = fw
    for _ in range(sa_steps):
        if deadline is not None and time.time() >= deadline:
            break
        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue
        a = int(perm[a_pos]); b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)
        accept = (d >= 0) or (T > 1e-12 and rng.random() < math.exp(d / max(T, 1e-12)))
        if accept:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)
            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()
        T *= alpha

    # MC greedy
    perm = best_perm.copy()
    rank = best_rank.copy()
    cur_fw = best_fw
    for _ in range(mc_steps):
        if deadline is not None and time.time() >= deadline:
            break
        a_pos = rng.randrange(n)
        b_pos = rng.randrange(n)
        if a_pos == b_pos:
            continue
        a = int(perm[a_pos]); b = int(perm[b_pos])
        d = delta_swap_fw(a, b, rank, out_nbrs, out_w, in_nbrs, in_w)
        if d > 0:
            perm[a_pos], perm[b_pos] = perm[b_pos], perm[a_pos]
            rank[a], rank[b] = rank[b], rank[a]
            cur_fw += float(d)
            if cur_fw > best_fw + 1e-9:
                best_fw = cur_fw
                best_perm = perm.copy()
                best_rank = rank.copy()

    return best_perm, best_rank, best_fw


# ============================================================
# 5) Stability loss + gradient wrt edge d
# ============================================================

def stability_loss_and_edge_dLd_d(
    n, eu, ev, ew, P, beta: float,
    loss_type: str,          # "none" | "hinge2" | "softplus"
    normalize: bool = True,
    eps_norm: float = 1e-12,
    margin: float = 0.0,
):
    if loss_type == "none":
        return 0.0, np.zeros_like(ew, dtype=np.float32)

    d = (P[ev] - P[eu]).astype(np.float32)
    x = (float(beta) * d).astype(np.float32)
    fwd = safe_sigmoid(x).astype(np.float32)
    bwd = (1.0 - fwd).astype(np.float32)

    BackIn  = np.zeros(n, dtype=np.float64)
    FwdOut  = np.zeros(n, dtype=np.float64)
    BackOut = np.zeros(n, dtype=np.float64)
    FwdIn   = np.zeros(n, dtype=np.float64)

    w64 = ew.astype(np.float64)
    np.add.at(BackIn,  ev, w64 * bwd.astype(np.float64))
    np.add.at(FwdOut,  eu, w64 * fwd.astype(np.float64))
    np.add.at(BackOut, eu, w64 * bwd.astype(np.float64))
    np.add.at(FwdIn,   ev, w64 * fwd.astype(np.float64))

    v1 = BackIn - FwdOut
    v2 = BackOut - FwdIn

    if normalize:
        Tot = np.zeros(n, dtype=np.float64)
        np.add.at(Tot, eu, w64)
        np.add.at(Tot, ev, w64)
        denom = Tot + float(eps_norm)
        v1n = v1 / denom
        v2n = v2 / denom
    else:
        denom = None
        v1n, v2n = v1, v2

    if loss_type == "hinge2":
        r1 = np.maximum(v1n, 0.0)
        r2 = np.maximum(v2n, 0.0)
        L = float(np.sum(r1 * r1) + np.sum(r2 * r2))
        dv1n = (2.0 * r1).astype(np.float64)
        dv2n = (2.0 * r2).astype(np.float64)
        if normalize:
            dv1 = dv1n / denom
            dv2 = dv2n / denom
        else:
            dv1, dv2 = dv1n, dv2n

    elif loss_type == "softplus":
        z1 = v1n - float(margin)
        z2 = v2n - float(margin)
        L = float(np.sum(softplus(z1)) + np.sum(softplus(z2)))
        s1 = safe_sigmoid(z1).astype(np.float64)
        s2 = safe_sigmoid(z2).astype(np.float64)
        if normalize:
            dv1 = (s1 / denom).astype(np.float64)
            dv2 = (s2 / denom).astype(np.float64)
        else:
            dv1, dv2 = s1, s2
    else:
        raise ValueError(f"Unknown stability loss_type={loss_type}")

    g_BackIn  = dv1
    g_FwdOut  = -dv1
    g_BackOut = dv2
    g_FwdIn   = -dv2

    df_dd = (float(beta) * (fwd * (1.0 - fwd))).astype(np.float32)
    db_dd = (-df_dd).astype(np.float32)

    edge_dLd_d = (
        (ew * db_dd) * (g_BackIn[ev].astype(np.float32) + g_BackOut[eu].astype(np.float32)) +
        (ew * df_dd) * (g_FwdOut[eu].astype(np.float32) + g_FwdIn[ev].astype(np.float32))
    ).astype(np.float32)

    return L, edge_dLd_d


# ============================================================
# 6) Rocket (FASTER default iters)
# ============================================================

def rocket_optimize_with_losses(
    n, eu, ev, ew, q_seke,
    lambda_seke: float = 0.0,
    stab_loss_type: str = "none",
    lambda_stab: float = 0.0,
    stab_normalize: bool = True,
    stab_margin: float = 0.0,
    beta: float = 1.0,
    iters: int = 120,      # DEFAULT smaller for speed
    lr: float = 0.05,
    seed: int = 1,
    deadline=None,
):
    rng = np.random.default_rng(seed)
    P = rng.standard_normal(n).astype(np.float32)

    wmax = float(np.max(ew)) if ew.size else 1.0
    if wmax <= 0:
        wmax = 1.0
    w_hat = (ew / wmax).astype(np.float32)

    lam = float(lambda_seke)
    lam_stab = float(lambda_stab)
    bet = float(beta)

    m = np.zeros(n, dtype=np.float32)
    v = np.zeros(n, dtype=np.float32)
    b1, b2 = 0.9, 0.999
    eps = 1e-8
    t = 0

    def perm_from_positions(Pvec):
        return np.argsort(Pvec, kind="mergesort").astype(np.int32)

    def rank_from_perm(perm):
        r = np.empty(n, dtype=np.int32)
        r[perm] = np.arange(n, dtype=np.int32)
        return r

    perm0 = perm_from_positions(P)
    rank0 = rank_from_perm(perm0)
    fw_best = float(np.sum(ew[rank0[eu] < rank0[ev]]))
    perm_best = perm0.copy()
    rank_best = rank0.copy()

    t0 = time.time()

    for _ in range(1, iters + 1):
        if deadline is not None and time.time() >= deadline:
            break

        d = (P[ev] - P[eu]).astype(np.float32)
        x = (bet * d).astype(np.float32)
        sig_f = safe_sigmoid(x).astype(np.float32)
        sigp = (sig_f * (1.0 - sig_f)).astype(np.float32)

        edge_grad_u = (w_hat * (1.0 - lam * q_seke) * sigp * bet).astype(np.float32)

        if stab_loss_type != "none" and lam_stab != 0.0:
            _, edge_dLd_d = stability_loss_and_edge_dLd_d(
                n=n, eu=eu, ev=ev, ew=ew, P=P, beta=bet,
                loss_type=stab_loss_type,
                normalize=stab_normalize,
                margin=stab_margin,
            )
            edge_grad_u = (edge_grad_u - lam_stab * edge_dLd_d).astype(np.float32)

        grad = np.zeros(n, dtype=np.float32)
        np.add.at(grad, eu, +edge_grad_u)
        np.add.at(grad, ev, -edge_grad_u)

        t += 1
        m = (b1 * m + (1 - b1) * grad).astype(np.float32)
        v = (b2 * v + (1 - b2) * (grad * grad)).astype(np.float32)
        mhat = m / (1 - (b1 ** t))
        vhat = v / (1 - (b2 ** t))
        P = (P - float(lr) * mhat / (np.sqrt(vhat) + eps)).astype(np.float32)

        # checkpoint best FW (discrete)
        perm = perm_from_positions(P)
        rank = rank_from_perm(perm)
        fw = float(np.sum(ew[rank[eu] < rank[ev]]))
        if fw > fw_best + 1e-9:
            fw_best = fw
            perm_best = perm.copy()
            rank_best = rank.copy()

    return perm_best, rank_best, fw_best, (time.time() - t0)


# ============================================================
# 7) One run config
# ============================================================

def run_one_config(
    edges_indexed, node_to_index, index_to_node,
    eu, ev, ew, out_nbrs, out_w, in_nbrs, in_w, q_seke,
    *,
    seed: int,
    time_limit_s,
    rocket_iters, rocket_beta, rocket_lr,
    lambda_seke, stab_loss_type, lambda_stab, stab_normalize, stab_margin,
    crane_sa_steps, crane_T0, crane_alpha, crane_mc_steps,
    write_ranking: bool,
    out_csv_path: str,
):
    t0 = time.time()
    deadline = (t0 + float(time_limit_s)) if (time_limit_s is not None and time_limit_s > 0) else None

    perm_r, rank_r, _, t_rocket = rocket_optimize_with_losses(
        n=len(node_to_index),
        eu=eu, ev=ev, ew=ew,
        q_seke=q_seke,
        lambda_seke=float(lambda_seke),
        stab_loss_type=str(stab_loss_type),
        lambda_stab=float(lambda_stab),
        stab_normalize=bool(stab_normalize),
        stab_margin=float(stab_margin),
        beta=float(rocket_beta),
        iters=int(rocket_iters),
        lr=float(rocket_lr),
        seed=int(seed),
        deadline=deadline,
    )
    tot1, fw1, bw1 = compute_forward_backward(edges_indexed, rank_r)

    t_cr0 = time.time()
    perm_c, rank_c, _ = crane_refine(
        perm_init=perm_r,
        eu=eu, ev=ev, ew=ew,
        out_nbrs=out_nbrs, out_w=out_w,
        in_nbrs=in_nbrs, in_w=in_w,
        seed=int(seed) + 1,
        sa_steps=int(crane_sa_steps),
        T0=float(crane_T0),
        alpha=float(crane_alpha),
        mc_steps=int(crane_mc_steps),
        deadline=deadline,
    )
    t_crane = time.time() - t_cr0
    tot2, fw2, bw2 = compute_forward_backward(edges_indexed, rank_c)

    if fw2 >= fw1 - 1e-9:
        final_rank = rank_c
        final_fw, final_bw, final_tot = fw2, bw2, tot2
        chosen = "rocket+crane"
    else:
        final_rank = rank_r
        final_fw, final_bw, final_tot = fw1, bw1, tot1
        chosen = "rocket_only"

    if write_ranking:
        write_ranking_csv_nodeid_order(out_csv_path, index_to_node, final_rank)

    t_total = time.time() - t0

    return {
        "seed": int(seed),
        "lambda_seke": float(lambda_seke),
        "stab_loss": str(stab_loss_type),
        "lambda_stab": float(lambda_stab),
        "stab_normalize": bool(stab_normalize),
        "stab_margin": float(stab_margin),
        "total_w": float(final_tot),
        "ratio_final_%": ratio_percent(float(final_fw), float(final_tot)),
        "ratio_after_rocket_%": ratio_percent(float(fw1), float(final_tot)),
        "ratio_after_crane_%": ratio_percent(float(fw2), float(final_tot)),
        "t_rocket_s": float(t_rocket),
        "t_crane_s": float(t_crane),
        "t_total_s": float(t_total),
        "chosen": chosen,
        "out_csv": out_csv_path if write_ranking else "",
    }


# ============================================================
# 8) Main: two-stage fast sweep
# ============================================================

if __name__ == "__main__":
    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d"

    # Outputs next to .d
    base_dir = os.path.dirname(os.path.abspath(edge_file))
    base = os.path.basename(edge_file)
    base = base[:-2] if base.endswith(".d") else base
    out_dir = os.path.join(base_dir, "seke_stability_fast_outputs")
    os.makedirs(out_dir, exist_ok=True)

    # FAST MODE budgets (edit here)
    TIME_LIMIT_S = None

    # fewer seeds for fast screening
    SEEDS_STAGE1 = [1, 2, 3]
    SEEDS_STAGE2 = [1, 2, 3]

    # faster Rocket/Crane
    ROCKET_ITERS = 120      # was 300
    ROCKET_BETA = 1.0
    ROCKET_LR = 0.05

    CRANE_SA_STEPS = 8000   # was 25000
    CRANE_T0 = 1.0
    CRANE_ALPHA = 0.98
    CRANE_MC_STEPS = 8000   # was 25000

    # Coarse sweep first
    LAM_SEKE_STAGE1 = [0.0, 0.1, 0.2, 0.5, 1.0]  # drop 0.05 and 2.0 for speed/safety
    LAM_STAB_STAGE2 = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
    STAB_LOSSES_STAGE2 = [("hinge2", True), ("softplus", True)]
    STAB_MARGIN = 0.0

    WRITE_RANKINGS = False
    PRINT_EVERY = 5

    print("\n================= FAST STABILITY SWEEP =================")
    print(f"Input:   {edge_file}")
    print(f"Out dir: {out_dir}")
    print(f"Budgets: rocket_iters={ROCKET_ITERS}, SA={CRANE_SA_STEPS}, MC={CRANE_MC_STEPS}")
    print("--------------------------------------------------------\n")

    # Load once
    t0 = time.time()
    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(edge_file)
    n = len(node_to_index)
    m = len(edges_indexed)

    eu = np.empty(m, dtype=np.int32)
    ev = np.empty(m, dtype=np.int32)
    ew = np.empty(m, dtype=np.float32)
    for i, (u, v, w) in enumerate(edges_indexed):
        eu[i] = u
        ev[i] = v
        ew[i] = float(w)

    out_nbrs, out_w, in_nbrs, in_w = build_inc_out_lists(n, eu, ev, ew)
    q_seke, tau_used = build_seke_edge_suspiciousness(n, eu, ev, ew, k=20.0, tau=None)
    print(f"[loaded] n={n} m={m} tau={tau_used:.6g} load_s={time.time()-t0:.2f}\n")

    all_rows = []
    rid = 0

    # ------------------------------
    # Stage 1: find best lambda_seke (stab=none)
    # ------------------------------
    print("Stage 1: SEKE-only sweep (stab=none)")
    stage1_rows = []
    total1 = len(SEEDS_STAGE1) * len(LAM_SEKE_STAGE1)

    for lam_seke in LAM_SEKE_STAGE1:
        for seed in SEEDS_STAGE1:
            rid += 1
            tag = f"stage1_stabnone_lamSEKE{fmt_lam(lam_seke)}"
            out_csv = os.path.join(out_dir, f"{base}_{tag}_seed{seed}.csv")
            res = run_one_config(
                edges_indexed, node_to_index, index_to_node,
                eu, ev, ew, out_nbrs, out_w, in_nbrs, in_w, q_seke,
                seed=seed,
                time_limit_s=TIME_LIMIT_S,
                rocket_iters=ROCKET_ITERS, rocket_beta=ROCKET_BETA, rocket_lr=ROCKET_LR,
                lambda_seke=lam_seke,
                stab_loss_type="none",
                lambda_stab=0.0,
                stab_normalize=False,
                stab_margin=STAB_MARGIN,
                crane_sa_steps=CRANE_SA_STEPS, crane_T0=CRANE_T0, crane_alpha=CRANE_ALPHA, crane_mc_steps=CRANE_MC_STEPS,
                write_ranking=WRITE_RANKINGS,
                out_csv_path=out_csv,
            )
            res["tag"] = tag
            stage1_rows.append(res)
            all_rows.append(res)

            if rid % PRINT_EVERY == 0 or rid == 1 or rid == total1:
                print(f"  [{rid:>3}/{total1}] seed={seed} lamSEKE={lam_seke} final={res['ratio_final_%']:.6f}%")

    df1 = pd.DataFrame(stage1_rows)
    best_lam = float(df1.groupby("lambda_seke")["ratio_final_%"].mean().sort_values(ascending=False).index[0])
    print(f"\nStage 1 best λ_SEKE (by mean over {SEEDS_STAGE1}): {best_lam}\n")

    # ------------------------------
    # Stage 2: stability sweep at fixed best lambda_seke
    # ------------------------------
    print("Stage 2: stability losses sweep at fixed λ_SEKE")
    stage2_rows = []
    total2 = len(SEEDS_STAGE2) * len(LAM_STAB_STAGE2) * len(STAB_LOSSES_STAGE2)

    done2 = 0
    for stab_loss, stab_norm in STAB_LOSSES_STAGE2:
        for lam_stab in LAM_STAB_STAGE2:
            for seed in SEEDS_STAGE2:
                done2 += 1
                tag = f"stage2_{stab_loss}_norm{int(stab_norm)}_lamSEKE{fmt_lam(best_lam)}_lamSTAB{fmt_lam(lam_stab)}"
                out_csv = os.path.join(out_dir, f"{base}_{tag}_seed{seed}.csv")

                res = run_one_config(
                    edges_indexed, node_to_index, index_to_node,
                    eu, ev, ew, out_nbrs, out_w, in_nbrs, in_w, q_seke,
                    seed=seed,
                    time_limit_s=TIME_LIMIT_S,
                    rocket_iters=ROCKET_ITERS, rocket_beta=ROCKET_BETA, rocket_lr=ROCKET_LR,
                    lambda_seke=best_lam,
                    stab_loss_type=stab_loss,
                    lambda_stab=lam_stab,
                    stab_normalize=stab_norm,
                    stab_margin=STAB_MARGIN,
                    crane_sa_steps=CRANE_SA_STEPS, crane_T0=CRANE_T0, crane_alpha=CRANE_ALPHA, crane_mc_steps=CRANE_MC_STEPS,
                    write_ranking=WRITE_RANKINGS,
                    out_csv_path=out_csv,
                )
                res["tag"] = tag
                stage2_rows.append(res)
                all_rows.append(res)

                if done2 % PRINT_EVERY == 0 or done2 == 1 or done2 == total2:
                    print(f"  [{done2:>3}/{total2}] seed={seed} {stab_loss} lamSTAB={lam_stab} final={res['ratio_final_%']:.6f}%")

    # Save CSVs
    df_all = pd.DataFrame(all_rows)
    all_csv = os.path.join(out_dir, f"{base}_FAST_ALL_RUNS.csv")
    df_all.to_csv(all_csv, index=False)

    # Aggregate
    grp_cols = ["tag", "stab_loss", "stab_normalize", "lambda_seke", "lambda_stab", "stab_margin"]
    agg = df_all.groupby(grp_cols).agg(
        mean_final_ratio=("ratio_final_%", "mean"),
        std_final_ratio=("ratio_final_%", "std"),
        mean_total_s=("t_total_s", "mean"),
        runs=("ratio_final_%", "count"),
    ).reset_index()
    agg_csv = os.path.join(out_dir, f"{base}_FAST_AGG.csv")
    agg.to_csv(agg_csv, index=False)

    # Print top configs (final)
    topk = agg.sort_values("mean_final_ratio", ascending=False).head(12)
    print("\n================= FAST SWEEP DONE =================")
    print(f"Saved:\n  {all_csv}\n  {agg_csv}\n")
    with pd.option_context("display.max_rows", 50, "display.max_columns", 50, "display.width", 180):
        print(topk[["stab_loss", "lambda_seke", "lambda_stab", "mean_final_ratio", "std_final_ratio", "runs", "mean_total_s"]].to_string(index=False))
    print("===================================================\n")

    print("NEXT:")
    print("  - Pick the best stability config above.")
    print("  - Re-run ONLY that config with 5 seeds + full budgets (rocket_iters=300, SA=25000, MC=25000) to confirm.\n")



================= STABILITY SWEEP (all outputs next to .d) =================
Input:   /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/connectome.d
Out dir: /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/seke_stability_sweep_outputs
Seeds:   [1, 2, 3, 4, 5]
Budgets: rocket_iters=300, SA=25000, MC=25000, time_limit=None
Lambda SEKE: [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
Lambda STAB: [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
Write rankings during sweep: False
-----------------------------------------------------------------------------

[loaded] n=136648 m=5657719  tau=0.004662  load+precompute_s=16.06

[   1/455] seed=1 stabnone_norm0_lamSEKE0_lamSTAB0 final=80.066659%  rocket_s=45.38  total_s=54.85
[  10/455] seed=5 stabnone_norm0_lamSEKE0p05_lamSTAB0 final=80.077823%  rocket_s=45.46  total_s=55.41
[  20/455] seed=5 stabnone_norm0_lamSEKE0p2_lamSTAB0 final=80.097228%  rocket_s=45.43  total_s=55.57
[  30/455] seed=5 stabnone_norm0_lamSEKE1_lamSTAB0 final=80.042890%  rocket_s=45.38  total_s=55.1

KeyboardInterrupt: 